# 1. 确认GPU

In [3]:
import torch

# 检查 MPS 是否可用
print("MPS 可用:", torch.backends.mps.is_available())
print("MPS 已构建支持:", torch.backends.mps.is_built())

# 查看当前 device
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("当前使用的设备:", device)

# 测试一下是否真的跑在 GPU
x = torch.randn(1000, 1000).to(device)
y = torch.matmul(x, x)
print("计算完成")

import subprocess

gpu_info = subprocess.check_output(
    ["system_profiler", "SPDisplaysDataType"]
).decode("utf-8")

print("GPU 信息:\n", gpu_info)

MPS 可用: True
MPS 已构建支持: True
当前使用的设备: mps
计算完成
GPU 信息:
 Graphics/Displays:

    Apple M1 Pro:

      Chipset Model: Apple M1 Pro
      Type: GPU
      Bus: Built-In
      Total Number of Cores: 14
      Vendor: Apple (0x106b)
      Metal Support: Metal 3
      Displays:
        U2790B:
          Resolution: 3840 x 2160 (2160p/4K UHD 1 - Ultra High Definition)
          UI Looks like: 1920 x 1080 @ 60.00Hz
          Main Display: Yes
          Mirror: Off
          Online: Yes
          Rotation: Supported
        Color LCD:
          Display Type: Built-in Liquid Retina XDR Display
          Resolution: 3024 x 1964 Retina
          Mirror: Off
          Online: Yes
          Automatically Adjust Brightness: No
          Connection Type: Internal




In [9]:
import torch
from torch import nn

torch.device('cpu'), torch.cuda.device('cuda'), torch.cuda.device('cuda:1'),torch.device('mps') # 使用 mac mps



(device(type='cpu'),
 device(type='mps'))

In [ ]:
print(torch.cuda.device_count())

# mac m1 pro 存在1个集成GPU
print(torch.mps.device_count())

0
1


In [21]:
def try_gpu(i=0):
    """如果存在，则返回gpu(i)，否则返回gpu"""
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

# mac m1 pro 只有一个mps 所以不需要i 参数，永远只会返回 mps:0
def try_mps():
    """如果存在，则返回mps(i)，否则返回cpu"""
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

def try_all_gpus():
    """返回所有可用的GPU，如果没有GPU，则返回[cpu(),]。"""
    devices = [torch.device(f'cuda:{i}') for i in range(torch.cuda.device_count())]      
    return devices if devices else [torch.device('cpu')]

def try_all_mps():
    """返回所有可用的MPS，如果没有MPS，则返回[cpu(),]。"""
    devices = [torch.device('mps')]
    return devices if torch.backends.mps.is_available() else [torch.device('cpu')]
                    
print(try_gpu())
print(try_gpu(10))
print(try_all_gpus())

print(try_mps())
print(try_all_mps())

cpu
cpu
[device(type='cpu')]
mps
[device(type='mps')]


In [22]:
# 查询张量所在设备
X = torch.tensor([1,2,3])
print(X.device) # 默认在CPU内存上

cpu


In [ ]:
# 存储在GPU上 mps
X = torch.ones(2,3,device=try_mps())
X 

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='mps:0')

In [24]:
# 在第二个GPU上创建一个随即张量 mps
Y = torch.rand(2,3,device=try_mps())
print(Y.device) # 没有1号GPU，则放到CPU上
Y

mps:0


tensor([[0.9911, 0.5693, 0.4833],
        [0.8714, 0.6587, 0.0877]], device='mps:0')

In [ ]:
# 要计算X+Y，我们需要决定在哪里执行这个操作 
# Z = X.cuda(0) # X+Y必须X和Y都在同一个GPU上 X.cuda(0) copy X到GPU0


tensor([[1., 1., 1.],
        [1., 1., 1.]], device='mps:0')

In [30]:
# Y = torch.rand(2,3,device=try_gpu(0))
# Y + Z
# Z.cuda(0) is Z # 如果变量在0号GPU时，就返回True

X + Y

tensor([[1.7727, 1.7177, 1.5069],
        [1.8858, 1.3054, 1.4532]], device='mps:0')

In [31]:
# 神经网络与GPU
# net = nn.Sequential(nn.Linear(3,1)) # 创建神经网络时已经把权重初始化好了
# net = net.to(device=try_gpu()) # 把所有参数在0号GPU上拷贝一份
# X = torch.ones(2,3,device=try_gpu()) # X 在0号GPU上
# net(X) # 所以前项运算所有元素都在0号GPU上运行

# 使用mps
net = nn.Sequential(nn.Linear(3,1)) # 创建神经网络时已经把权重初始化好了
net = net.to(device=try_mps()) # 把所有参数在0号MPS上拷贝一份
X = torch.ones(2,3,device=try_mps()) # X 在0号MPS上
net(X) # 所以前项运算所有元素都在0号MPS上运行

tensor([[-1.0513],
        [-1.0513]], device='mps:0', grad_fn=<LinearBackward0>)

In [32]:
# 确认模型参数存储在同一个GPU、mps上
net[0].weight.data.device

device(type='mps', index=0)